# Small vs frontier: quality vs cost vs latency

**Session 7 · small model (`llama3.2:3b`) vs big model (`gpt-oss:120b-cloud`)**

Run one task on both models and lay the numbers side by side: accuracy, latency, and a
cost-per-1000-calls estimate. Two tasks, opposite conclusions — that is the whole point.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
import re
import time
from utils import ask, count_tokens, SMALL_MODEL, BIG_MODEL
from eval import load_cases, run_eval, exact


### Worked example

A `benchmark()` helper: accuracy (from the eval harness), mean latency, mean output tokens,
and a cost-per-1000-calls figure at each model's rate. Run it on an **easy** task
(sentiment) and a **hard** one (multi-step math).

In [ ]:
# $/1M tokens (input, output). SMALL runs locally -- "cost" is compute you already own;
# we still price it at a small self-hosting estimate so the columns are comparable.
RATES = {
    SMALL_MODEL: (0.05, 0.10),
    BIG_MODEL:   (0.60, 2.50),
}

def benchmark(cases, build_prompt, extract, model, scorer, repeats=1):
    lat, out_tok = [], []

    def timed(x):
        t0 = time.time()
        raw = ask(build_prompt(x), model=model)
        lat.append(time.time() - t0)
        out_tok.append(count_tokens(raw))
        return extract(raw)

    rep = run_eval(cases, timed, scorer=scorer, repeats=repeats)
    in_tok = sum(count_tokens(build_prompt(c["input"])) for c in cases) / len(cases)
    mean_out = sum(out_tok) / len(out_tok)
    pin, pout = RATES.get(model, (0, 0))
    cost_1k = (in_tok * pin + mean_out * pout) / 1_000_000 * 1000
    print(f"  {model:<22} acc {rep['acc_mean']:.0%}  "
          f"lat {sum(lat) / len(lat):4.1f}s  out ~{mean_out:3.0f} tok  ~${cost_1k:.3f}/1k calls")
    return rep

# --- easy task: sentiment ---
LABS = ["positive", "negative", "neutral"]
sent = load_cases("../eval/datasets/sentiment.jsonl")
sent_prompt = lambda t: f'Classify sentiment as positive, negative, or neutral. One word.\nText: "{t}" ->'
sent_extract = lambda o: next((l for l in LABS if l in o.strip().lower()), o.strip().lower())

print(f"EASY TASK - sentiment (n={len(sent)}):")
for m in (SMALL_MODEL, BIG_MODEL):
    try:
        benchmark(sent, sent_prompt, sent_extract, m, exact)
    except Exception as e:
        print(f"  {m:<22} skipped: {type(e).__name__}")


### The hard task flips the conclusion

Same three columns, a task that needs reasoning. Now the big model's accuracy lead should be
wide enough to justify its cost and latency.

In [ ]:
# --- hard task: multi-step math ---
math = load_cases("../eval/datasets/math_reasoning.jsonl")
math_prompt = lambda q: f"{q}\nThink step by step, then end with 'FINAL: <number>'."

def math_extract(o):
    nums = re.findall(r"-?\d+(?:\.\d+)?", o.split("FINAL:")[-1].replace(",", ""))
    return nums[-1] if nums else o.strip()

def num_match(o, e):
    try:
        return abs(float(o) - float(e)) < 1e-6
    except ValueError:
        return False

print(f"HARD TASK - multi-step math (n={len(math)}):")
for m in (SMALL_MODEL, BIG_MODEL):
    try:
        benchmark(math, math_prompt, math_extract, m, num_match)
    except Exception as e:
        print(f"  {m:<22} skipped: {type(e).__name__}")

## Your turn - vary the example

1. On the easy task the small model should match the big one at a fraction of the cost and
   latency; on the hard task the big model should win accuracy by a wide margin. Write the
   one-line routing rule this implies.
2. Add a middle task (structured extraction from notebook 04/02). Which side of the line does
   it fall on?
3. Try "route by difficulty": small model first, escalate to big only when the small model's
   answer fails a cheap check. Estimate the blended cost per 1000 calls.